# MABe Inference Notebook 

In [ ]:
# MABe 推理流程：读取测试集、构造特征、加载模型并生成提交文件

from pathlib import Path
import os
import sys
COMPETITION_ROOT = Path('/kaggle/input/MABe-mouse-behavior-detection')
TRAINED_MODEL_DIR = Path('/kaggle/input/results-v1-21')
METRIC_PACKAGE_DIR = Path('/kaggle/input/mabe-package')
if not COMPETITION_ROOT.exists():
    raise FileNotFoundError('Competition dataset not found!')
if not TRAINED_MODEL_DIR.exists():
    raise FileNotFoundError('Model dataset not found! 请先将训练notebook的output保存为dataset，然后添加到这个notebook的input中')
import gc
import re
import ast
import itertools
import json
from pathlib import Path
import numpy as np
import pandas as pd
import polars as pl
import xgboost as xgb
import lightgbm as lgb
from tqdm.auto import tqdm
DATA_ROOT = COMPETITION_ROOT
TEST_POSE_DIR = DATA_ROOT / 'test_tracking'
ARTIFACT_DIR = Path('/kaggle/working')
SELF_FEATURE_OUTPUT_DIR = ARTIFACT_DIR / 'self_features'
PAIR_FEATURE_OUTPUT_DIR = ARTIFACT_DIR / 'pair_features'
SELF_FEATURE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PAIR_FEATURE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
KEY_COLUMNS = ['video_id', 'agent_mouse_id', 'target_mouse_id', 'video_frame']
KEYPOINT_NAMES = ['ear_left', 'ear_right', 'nose', 'neck', 'body_center', 'lateral_left', 'lateral_right', 'hip_left', 'hip_right', 'tail_base', 'tail_tip']
SELF_ACTIONS = ['biteobject', 'climb', 'dig', 'exploreobject', 'freeze', 'genitalgroom', 'huddle', 'rear', 'rest', 'run', 'selfgroom']
PAIR_ACTIONS = ['allogroom', 'approach', 'attack', 'attemptmount', 'avoid', 'chase', 'chaseattack', 'defend', 'disengage', 'dominance', 'dominancegroom', 'dominancemount', 'ejaculate', 'escape', 'flinch', 'follow', 'intromit', 'mount', 'reciprocalsniff', 'shepherd', 'sniff', 'sniffbody', 'sniffface', 'sniffgenital', 'submit', 'tussle']
scales = np.logspace(np.log10(50), np.log10(5000), num=7, dtype=int).tolist()
MODEL_CONFIG = {'speed_time_scales': scales, 'disp_time_scales': scales, 'use_ensemble': True, 'smoothing_window': 5, 'min_duration': 1}

# 构造单只小鼠的姿态、速度、角度和时间上下文特征。
def build_self_mouse_features(video_meta: dict, pose_df: pl.DataFrame) -> pl.DataFrame:
    """
    增强版self特征:
    - 多时间尺度速度
    - 加速度特征
    - 位移特征
    - 身体角度变化率
    - 伸长度变化率
    - 新增: 姿态特征、运动模式特征、静止检测
    """

    def add_part_distance(keypoint_a, keypoint_b):
        return ((pl.col(f'agent_x_{keypoint_a}') - pl.col(f'agent_x_{keypoint_b}')).pow(2) + (pl.col(f'agent_y_{keypoint_a}') - pl.col(f'agent_y_{keypoint_b}')).pow(2)).sqrt() / video_meta['pix_per_cm_approx']

    def add_part_speed(keypoint_name, time_lag_ms):
        frame_window = max(1, int(round(time_lag_ms * video_meta['frames_per_second'] / 1000.0)))
        return ((pl.col(f'agent_x_{keypoint_name}').diff().pow(2) + pl.col(f'agent_y_{keypoint_name}').diff().pow(2)).sqrt() / video_meta['pix_per_cm_approx'] * video_meta['frames_per_second']).rolling_mean(window_size=frame_window, center=True, min_samples=1)

    def add_displacement(keypoint_name, time_lag_ms):
        """一段时间内的总位移"""
        frame_window = max(1, int(round(time_lag_ms * video_meta['frames_per_second'] / 1000.0)))
        return ((pl.col(f'agent_x_{keypoint_name}') - pl.col(f'agent_x_{keypoint_name}').shift(frame_window)).pow(2) + (pl.col(f'agent_y_{keypoint_name}') - pl.col(f'agent_y_{keypoint_name}').shift(frame_window)).pow(2)).sqrt() / video_meta['pix_per_cm_approx']

    def add_body_elongation():
        distance_left = add_part_distance('nose', 'tail_base')
        distance_right = add_part_distance('ear_left', 'ear_right')
        return distance_left / (distance_right + 1e-06)

    def add_body_angle():
        v1x = pl.col('agent_x_nose') - pl.col('agent_x_body_center')
        v1y = pl.col('agent_y_nose') - pl.col('agent_y_body_center')
        v2x = pl.col('agent_x_tail_base') - pl.col('agent_x_body_center')
        v2y = pl.col('agent_y_tail_base') - pl.col('agent_y_body_center')
        return (v1x * v2x + v1y * v2y) / ((v1x.pow(2) + v1y.pow(2)).sqrt() * (v2x.pow(2) + v2y.pow(2)).sqrt() + 1e-06)

    def add_acceleration(keypoint_name, time_lag_ms):
        """加速度: 速度的变化率"""
        frame_window = max(1, int(round(time_lag_ms * video_meta['frames_per_second'] / 1000.0)))
        speed_values = (pl.col(f'agent_x_{keypoint_name}').diff().pow(2) + pl.col(f'agent_y_{keypoint_name}').diff().pow(2)).sqrt() / video_meta['pix_per_cm_approx'] * video_meta['frames_per_second']
        return speed_values.diff().rolling_mean(window_size=frame_window, center=True, min_samples=1)

    def add_speed_std(keypoint_name, time_lag_ms):
        """速度的标准差 - 捕捉运动的不稳定性"""
        frame_window = max(1, int(round(time_lag_ms * video_meta['frames_per_second'] / 1000.0)))
        speed_values = (pl.col(f'agent_x_{keypoint_name}').diff().pow(2) + pl.col(f'agent_y_{keypoint_name}').diff().pow(2)).sqrt() / video_meta['pix_per_cm_approx'] * video_meta['frames_per_second']
        return speed_values.rolling_std(window_size=frame_window, center=True, min_samples=1)

    def add_height_proxy(keypoint_name):
        """
        身体部位的'高度'代理 - 用y坐标的相对位置
        对于rear（站立）行为，nose的y坐标会相对较小（图像坐标系）
        """
        return pl.col(f'agent_y_{keypoint_name}') - pl.col('agent_y_body_center')

    def add_vertical_extension():
        """
        垂直伸展度: nose和tail_base的y坐标差
        rear时这个值会变大（nose向上）
        """
        return (pl.col('agent_y_tail_base') - pl.col('agent_y_nose')) / video_meta['pix_per_cm_approx']

    def add_body_compactness():
        """
        身体紧凑度: 所有部位到body_center的平均距离
        huddle/freeze时身体会更紧凑
        """
        mouse_parts = ['nose', 'ear_left', 'ear_right', 'tail_base', 'hip_left', 'hip_right']
        distance_list = []
        for keypoint_part in mouse_parts:
            distance_values = ((pl.col(f'agent_x_{keypoint_part}') - pl.col('agent_x_body_center')).pow(2) + (pl.col(f'agent_y_{keypoint_part}') - pl.col('agent_y_body_center')).pow(2)).sqrt() / video_meta['pix_per_cm_approx']
            distance_list.append(distance_values)
        return sum(distance_list) / len(distance_list)

    def add_tail_curvature():
        """
        尾巴弯曲度: tail_base到tail_tip的距离与body_center到tail_tip的距离之比
        """
        distance_left = add_part_distance('tail_base', 'tail_tip')
        distance_right = add_part_distance('body_center', 'tail_tip')
        return distance_left / (distance_right + 1e-06)

    def add_head_body_angle():
        """
        头部相对身体的角度
        用于检测grooming（头转向身体）
        """
        ear_mid_x = (pl.col('agent_x_ear_left') + pl.col('agent_x_ear_right')) / 2
        ear_mid_y = (pl.col('agent_y_ear_left') + pl.col('agent_y_ear_right')) / 2
        head_dir_x = pl.col('agent_x_nose') - ear_mid_x
        head_dir_y = pl.col('agent_y_nose') - ear_mid_y
        body_dir_x = pl.col('agent_x_tail_base') - pl.col('agent_x_body_center')
        body_dir_y = pl.col('agent_y_tail_base') - pl.col('agent_y_body_center')
        dot_product = head_dir_x * body_dir_x + head_dir_y * body_dir_y
        vector_norm_a = (head_dir_x.pow(2) + head_dir_y.pow(2)).sqrt()
        vector_norm_b = (body_dir_x.pow(2) + body_dir_y.pow(2)).sqrt()
        return dot_product / (vector_norm_a * vector_norm_b + 1e-06)

    def add_nose_body_distance():
        """
        nose到身体各部位的距离 - 用于检测selfgroom
        """
        return add_part_distance('nose', 'hip_left') + add_part_distance('nose', 'hip_right')

    def add_angular_velocity(time_lag_ms):
        """
        身体朝向的角速度
        用于检测旋转/转向行为
        """
        frame_window = max(1, int(round(time_lag_ms * video_meta['frames_per_second'] / 1000.0)))
        dir_x = pl.col('agent_x_nose') - pl.col('agent_x_tail_base')
        dir_y = pl.col('agent_y_nose') - pl.col('agent_y_tail_base')
        angle = pl.arctan2(dir_y, dir_x)
        return angle.diff().rolling_mean(window_size=frame_window, center=True, min_samples=1)

    def add_movement_regularity(keypoint_name, time_lag_ms):
        """
        运动规律性: 位移与路径长度的比值
        值接近1表示直线运动，接近0表示原地打转
        用于区分run vs dig
        """
        frame_window = max(1, int(round(time_lag_ms * video_meta['frames_per_second'] / 1000.0)))
        net_disp = add_displacement(keypoint_name, time_lag_ms)
        frame_disp = (pl.col(f'agent_x_{keypoint_name}').diff().pow(2) + pl.col(f'agent_y_{keypoint_name}').diff().pow(2)).sqrt() / video_meta['pix_per_cm_approx']
        path_length = frame_disp.rolling_sum(window_size=frame_window, center=True, min_samples=1)
        return net_disp / (path_length + 1e-06)

    def add_stillness_ratio(time_lag_ms, still_speed_threshold=1.0):
        """
        静止比例: 在时间窗口内速度低于阈值的帧数占比
        用于检测freeze, rest
        """
        frame_window = max(1, int(round(time_lag_ms * video_meta['frames_per_second'] / 1000.0)))
        speed_values = (pl.col('agent_x_body_center').diff().pow(2) + pl.col('agent_y_body_center').diff().pow(2)).sqrt() / video_meta['pix_per_cm_approx'] * video_meta['frames_per_second']
        is_still = (speed_values < still_speed_threshold).cast(pl.Float32)
        return is_still.rolling_mean(window_size=frame_window, center=True, min_samples=1)

    def add_limb_asymmetry():
        """
        肢体不对称性: 左右hip/lateral的距离差
        可能与特定行为姿态相关
        """
        left_dist = add_part_distance('body_center', 'hip_left')
        right_dist = add_part_distance('body_center', 'hip_right')
        return (left_dist - right_dist).abs()

    def add_elongation_change_rate(time_lag_ms):
        """伸长度变化率 - 捕捉身体伸缩"""
        frame_window = max(1, int(round(time_lag_ms * video_meta['frames_per_second'] / 1000.0)))
        elong = add_body_elongation()
        return elong.diff().rolling_mean(window_size=frame_window, center=True, min_samples=1)

    def add_body_angle_change_rate(time_lag_ms):
        """身体弯曲角度变化率"""
        frame_window = max(1, int(round(time_lag_ms * video_meta['frames_per_second'] / 1000.0)))
        angle = add_body_angle()
        return angle.diff().rolling_mean(window_size=frame_window, center=True, min_samples=1)
    mouse_count = (video_meta['mouse1_strain'] is not None) + (video_meta['mouse2_strain'] is not None) + (video_meta['mouse3_strain'] is not None) + (video_meta['mouse4_strain'] is not None)
    start_frame_id = pose_df.select(pl.col('video_frame').min()).item()
    end_frame_id = pose_df.select(pl.col('video_frame').max()).item()
    result_df = []
    wide_pose_df = pose_df.pivot(on=['bodypart'], index=['video_frame', 'mouse_id'], values=['x', 'y']).sort(['mouse_id', 'video_frame'])
    wide_pose_tracks = {mouse_id: wide_pose_df.filter(pl.col('mouse_id') == mouse_id) for mouse_id in range(1, mouse_count + 1)}
    speed_keypoints = ['ear_left', 'ear_right', 'tail_base', 'body_center', 'nose']
    for agent_id in range(1, mouse_count + 1):
        segment_record = pl.DataFrame({'video_id': video_meta['video_id'], 'agent_mouse_id': agent_id, 'target_mouse_id': -1, 'video_frame': pl.arange(start_frame_id, end_frame_id + 1, eager=True)}, schema={'video_id': pl.Int32, 'agent_mouse_id': pl.Int8, 'target_mouse_id': pl.Int8, 'video_frame': pl.Int32})
        pivot_data = wide_pose_tracks[agent_id].select(pl.col('video_frame'), pl.exclude('video_frame').name.prefix('agent_'))
        available_columns = pivot_data.columns
        pivot_data = pivot_data.with_columns(*[pl.lit(None).cast(pl.Float32).alias(f'agent_x_{keypoint}') for keypoint in KEYPOINT_NAMES if f'agent_x_{keypoint}' not in available_columns], *[pl.lit(None).cast(pl.Float32).alias(f'agent_y_{keypoint}') for keypoint in KEYPOINT_NAMES if f'agent_y_{keypoint}' not in available_columns])
        feature_expressions = [pl.col('video_frame'), pl.lit(agent_id).alias('agent_mouse_id'), pl.lit(-1).alias('target_mouse_id')]
        feature_expressions.extend([add_part_distance(keypoint_a, keypoint_b).alias(f'aa__{keypoint_a}__{keypoint_b}__distance') for keypoint_a, keypoint_b in itertools.combinations(KEYPOINT_NAMES, 2)])
        feature_expressions.extend([add_part_speed(keypoint_name, time_lag_ms).alias(f'agent__{keypoint_name}__speed_{time_lag_ms}ms') for keypoint_name, time_lag_ms in itertools.product(speed_keypoints, MODEL_CONFIG['speed_time_scales'])])
        feature_expressions.extend([add_displacement('body_center', time_lag_ms).alias(f'agent__body_center__disp_{time_lag_ms}ms') for time_lag_ms in MODEL_CONFIG['disp_time_scales']])
        feature_expressions.append(add_body_elongation().alias('agent__elongation'))
        feature_expressions.append(add_body_angle().alias('agent__body_angle'))
        for time_lag_ms in scales:
            feature_expressions.append(add_acceleration('body_center', time_lag_ms).alias(f'agent__body_center__accel_{time_lag_ms}ms'))
        for time_lag_ms in scales:
            feature_expressions.append(add_speed_std('body_center', time_lag_ms).alias(f'agent__body_center__speed_std_{time_lag_ms}ms'))
        feature_expressions.append(add_vertical_extension().alias('agent__vertical_extension'))
        feature_expressions.append(add_body_compactness().alias('agent__body_compactness'))
        feature_expressions.append(add_tail_curvature().alias('agent__tail_curvature'))
        feature_expressions.append(add_head_body_angle().alias('agent__head_body_angle'))
        feature_expressions.append(add_nose_body_distance().alias('agent__nose_to_body_dist'))
        for time_lag_ms in scales:
            feature_expressions.append(add_angular_velocity(time_lag_ms).alias(f'agent__angular_velocity_{time_lag_ms}ms'))
        for time_lag_ms in scales:
            feature_expressions.append(add_movement_regularity('body_center', time_lag_ms).alias(f'agent__movement_regularity_{time_lag_ms}ms'))
        for time_lag_ms in scales:
            feature_expressions.append(add_stillness_ratio(time_lag_ms, still_speed_threshold=1.0).alias(f'agent__stillness_ratio_{time_lag_ms}ms'))
        feature_expressions.append(add_limb_asymmetry().alias('agent__limb_asymmetry'))
        for time_lag_ms in scales:
            feature_expressions.append(add_elongation_change_rate(time_lag_ms).alias(f'agent__elongation_change_{time_lag_ms}ms'))
        for time_lag_ms in scales:
            feature_expressions.append(add_body_angle_change_rate(time_lag_ms).alias(f'agent__body_angle_change_{time_lag_ms}ms'))
        for time_lag_ms in scales:
            nose_speed = add_part_speed('nose', time_lag_ms)
            body_speed = add_part_speed('body_center', time_lag_ms)
            feature_expressions.append((nose_speed / (body_speed + 1e-06)).alias(f'agent__nose_body_speed_ratio_{time_lag_ms}ms'))
            tail_speed = add_part_speed('tail_base', time_lag_ms)
            feature_expressions.append((tail_speed / (body_speed + 1e-06)).alias(f'agent__tail_body_speed_ratio_{time_lag_ms}ms'))
        for time_lag_ms in scales:
            frame_window = max(1, int(round(time_lag_ms * video_meta['frames_per_second'] / 1000.0)))
            dx = pl.col('agent_x_body_center').diff()
            dy = pl.col('agent_y_body_center').diff()
            direction_change = dx * dx.shift(1) + dy * dy.shift(1) < 0
            feature_expressions.append(direction_change.cast(pl.Float32).rolling_mean(window_size=frame_window, center=True, min_samples=1).alias(f'agent__direction_reversal_ratio_{time_lag_ms}ms'))
        feature_columns = pivot_data.with_columns(pl.lit(agent_id).alias('agent_mouse_id'), pl.lit(-1).alias('target_mouse_id')).select(feature_expressions)
        segment_record = segment_record.join(feature_columns, on=['video_frame', 'agent_mouse_id', 'target_mouse_id'], how='left')
        result_df.append(segment_record)
    return pl.concat(result_df, how='vertical')

# 构造两只小鼠之间的距离、相对运动和交互方向特征。
def build_pair_mouse_features(video_meta: dict, pose_df: pl.DataFrame) -> pl.DataFrame:
    """
    增强版pair特征:
    - agent-target部位间距离
    - 多时间尺度速度
    - 相对速度
    - 接近/远离指标
    - 距离变化率 (新增)
    """

    def add_part_distance(mouse_role_a, keypoint_a, mouse_role_b, keypoint_b):
        return ((pl.col(f'{mouse_role_a}_x_{keypoint_a}') - pl.col(f'{mouse_role_b}_x_{keypoint_b}')).pow(2) + (pl.col(f'{mouse_role_a}_y_{keypoint_a}') - pl.col(f'{mouse_role_b}_y_{keypoint_b}')).pow(2)).sqrt() / video_meta['pix_per_cm_approx']

    def add_part_speed(mouse_role, keypoint_name, time_lag_ms):
        frame_window = max(1, int(round(time_lag_ms * video_meta['frames_per_second'] / 1000.0)))
        return ((pl.col(f'{mouse_role}_x_{keypoint_name}').diff().pow(2) + pl.col(f'{mouse_role}_y_{keypoint_name}').diff().pow(2)).sqrt() / video_meta['pix_per_cm_approx'] * video_meta['frames_per_second']).rolling_mean(window_size=frame_window, center=True, min_samples=1)

    def add_body_elongation(mouse_role):
        distance_left = add_part_distance(mouse_role, 'nose', mouse_role, 'tail_base')
        distance_right = add_part_distance(mouse_role, 'ear_left', mouse_role, 'ear_right')
        return distance_left / (distance_right + 1e-06)

    def add_body_angle(mouse_role):
        v1x = pl.col(f'{mouse_role}_x_nose') - pl.col(f'{mouse_role}_x_body_center')
        v1y = pl.col(f'{mouse_role}_y_nose') - pl.col(f'{mouse_role}_y_body_center')
        v2x = pl.col(f'{mouse_role}_x_tail_base') - pl.col(f'{mouse_role}_x_body_center')
        v2y = pl.col(f'{mouse_role}_y_tail_base') - pl.col(f'{mouse_role}_y_body_center')
        return (v1x * v2x + v1y * v2y) / ((v1x.pow(2) + v1y.pow(2)).sqrt() * (v2x.pow(2) + v2y.pow(2)).sqrt() + 1e-06)

    def add_center_distance_rolling(agg, time_lag_ms):
        expr = add_part_distance('agent', 'body_center', 'target', 'body_center')
        frame_window = max(1, int(round(time_lag_ms * video_meta['frames_per_second'] / 1000.0)))
        if agg == 'mean':
            return expr.rolling_mean(window_size=frame_window, center=True, min_samples=1)
        elif agg == 'std':
            return expr.rolling_std(window_size=frame_window, center=True, min_samples=1)
        elif agg == 'min':
            return expr.rolling_min(window_size=frame_window, center=True, min_samples=1)
        elif agg == 'max':
            return expr.rolling_max(window_size=frame_window, center=True, min_samples=1)
        else:
            raise ValueError()

    def add_distance_change_rate(keypoint_a, keypoint_b, time_lag_ms):
        """
        计算两个部位之间距离的变化率
        正值 = 远离, 负值 = 接近
        单位: cm/s
        """
        frame_window = max(1, int(round(time_lag_ms * video_meta['frames_per_second'] / 1000.0)))
        distance_values = add_part_distance('agent', keypoint_a, 'target', keypoint_b)
        dist_change = (distance_values - distance_values.shift(frame_window)) / (frame_window / video_meta['frames_per_second'])
        return dist_change

    def add_distance_difference(keypoint_a, keypoint_b):
        """
        帧间距离变化 (用于后续rolling计算)
        """
        distance_values = add_part_distance('agent', keypoint_a, 'target', keypoint_b)
        return distance_values.diff() * video_meta['frames_per_second']

    def add_distance_change_rolling(keypoint_a, keypoint_b, time_lag_ms, agg='mean'):
        """
        距离变化的滚动统计
        """
        frame_window = max(1, int(round(time_lag_ms * video_meta['frames_per_second'] / 1000.0)))
        dist_diff = add_distance_difference(keypoint_a, keypoint_b)
        if agg == 'mean':
            return dist_diff.rolling_mean(window_size=frame_window, center=True, min_samples=1)
        elif agg == 'std':
            return dist_diff.rolling_std(window_size=frame_window, center=True, min_samples=1)
        elif agg == 'min':
            return dist_diff.rolling_min(window_size=frame_window, center=True, min_samples=1)
        elif agg == 'max':
            return dist_diff.rolling_max(window_size=frame_window, center=True, min_samples=1)
        else:
            raise ValueError()

    def add_approaching_ratio(keypoint_a, keypoint_b, time_lag_ms):
        """
        计算在一段时间窗口内，接近的帧数占比
        值域 [0, 1], 接近1表示一直在接近
        """
        frame_window = max(1, int(round(time_lag_ms * video_meta['frames_per_second'] / 1000.0)))
        distance_values = add_part_distance('agent', keypoint_a, 'target', keypoint_b)
        is_approaching = (distance_values.diff() < 0).cast(pl.Float32)
        return is_approaching.rolling_mean(window_size=frame_window, center=True, min_samples=1)

    def add_agent_facing_target():
        """agent是否面向target"""
        agent_dir_x = pl.col('agent_x_nose') - pl.col('agent_x_body_center')
        agent_dir_y = pl.col('agent_y_nose') - pl.col('agent_y_body_center')
        to_target_x = pl.col('target_x_body_center') - pl.col('agent_x_body_center')
        to_target_y = pl.col('target_y_body_center') - pl.col('agent_y_body_center')
        dot_product = agent_dir_x * to_target_x + agent_dir_y * to_target_y
        vector_norm_a = (agent_dir_x.pow(2) + agent_dir_y.pow(2)).sqrt()
        vector_norm_b = (to_target_x.pow(2) + to_target_y.pow(2)).sqrt()
        return dot_product / (vector_norm_a * vector_norm_b + 1e-06)

    def add_target_facing_agent():
        """target是否面向agent"""
        target_dir_x = pl.col('target_x_nose') - pl.col('target_x_body_center')
        target_dir_y = pl.col('target_y_nose') - pl.col('target_y_body_center')
        to_agent_x = pl.col('agent_x_body_center') - pl.col('target_x_body_center')
        to_agent_y = pl.col('agent_y_body_center') - pl.col('target_y_body_center')
        dot_product = target_dir_x * to_agent_x + target_dir_y * to_agent_y
        vector_norm_a = (target_dir_x.pow(2) + target_dir_y.pow(2)).sqrt()
        vector_norm_b = (to_agent_x.pow(2) + to_agent_y.pow(2)).sqrt()
        return dot_product / (vector_norm_a * vector_norm_b + 1e-06)

    def add_agent_behind_target():
        """agent是否在target后面"""
        target_dir_x = pl.col('target_x_nose') - pl.col('target_x_tail_base')
        target_dir_y = pl.col('target_y_nose') - pl.col('target_y_tail_base')
        to_agent_x = pl.col('agent_x_body_center') - pl.col('target_x_body_center')
        to_agent_y = pl.col('agent_y_body_center') - pl.col('target_y_body_center')
        dot_product = target_dir_x * to_agent_x + target_dir_y * to_agent_y
        vector_norm_a = (target_dir_x.pow(2) + target_dir_y.pow(2)).sqrt()
        vector_norm_b = (to_agent_x.pow(2) + to_agent_y.pow(2)).sqrt()
        return -dot_product / (vector_norm_a * vector_norm_b + 1e-06)

    def add_agent_nose_to_rear():
        """agent的nose是否更靠近target的后部"""
        nose_to_tail = add_part_distance('agent', 'nose', 'target', 'tail_base')
        nose_to_nose = add_part_distance('agent', 'nose', 'target', 'nose')
        return nose_to_nose / (nose_to_tail + 1e-06)

    def add_relative_speed(time_lag_ms):
        """相对速度"""
        agent_speed = add_part_speed('agent', 'body_center', time_lag_ms)
        target_speed = add_part_speed('target', 'body_center', time_lag_ms)
        return agent_speed - target_speed

    def add_speed_ratio(time_lag_ms):
        """速度比值"""
        agent_speed = add_part_speed('agent', 'body_center', time_lag_ms)
        target_speed = add_part_speed('target', 'body_center', time_lag_ms)
        return agent_speed / (target_speed + 1e-06)
    mouse_count = (video_meta['mouse1_strain'] is not None) + (video_meta['mouse2_strain'] is not None) + (video_meta['mouse3_strain'] is not None) + (video_meta['mouse4_strain'] is not None)
    start_frame_id = pose_df.select(pl.col('video_frame').min()).item()
    end_frame_id = pose_df.select(pl.col('video_frame').max()).item()
    result_df = []
    wide_pose_df = pose_df.pivot(on=['bodypart'], index=['video_frame', 'mouse_id'], values=['x', 'y']).sort(['mouse_id', 'video_frame'])
    wide_pose_tracks = {mouse_id: wide_pose_df.filter(pl.col('mouse_id') == mouse_id) for mouse_id in range(1, mouse_count + 1)}
    speed_keypoints = ['ear_left', 'ear_right', 'tail_base', 'body_center']
    dist_change_time_scales = scales
    for agent_id, target_id in itertools.permutations(range(1, mouse_count + 1), 2):
        segment_record = pl.DataFrame({'video_id': video_meta['video_id'], 'agent_mouse_id': agent_id, 'target_mouse_id': target_id, 'video_frame': pl.arange(start_frame_id, end_frame_id + 1, eager=True)}, schema={'video_id': pl.Int32, 'agent_mouse_id': pl.Int8, 'target_mouse_id': pl.Int8, 'video_frame': pl.Int32})
        merged_pivot = wide_pose_tracks[agent_id].select(pl.col('video_frame'), pl.exclude('video_frame').name.prefix('agent_')).join(wide_pose_tracks[target_id].select(pl.col('video_frame'), pl.exclude('video_frame').name.prefix('target_')), on='video_frame', how='inner')
        available_columns = merged_pivot.columns
        merged_pivot = merged_pivot.with_columns(*[pl.lit(None).cast(pl.Float32).alias(f'agent_x_{keypoint}') for keypoint in KEYPOINT_NAMES if f'agent_x_{keypoint}' not in available_columns], *[pl.lit(None).cast(pl.Float32).alias(f'agent_y_{keypoint}') for keypoint in KEYPOINT_NAMES if f'agent_y_{keypoint}' not in available_columns], *[pl.lit(None).cast(pl.Float32).alias(f'target_x_{keypoint}') for keypoint in KEYPOINT_NAMES if f'target_x_{keypoint}' not in available_columns], *[pl.lit(None).cast(pl.Float32).alias(f'target_y_{keypoint}') for keypoint in KEYPOINT_NAMES if f'target_y_{keypoint}' not in available_columns])
        feature_expressions = [pl.col('video_frame'), pl.lit(agent_id).alias('agent_mouse_id'), pl.lit(target_id).alias('target_mouse_id')]
        feature_expressions.extend([add_part_distance('agent', agent_body_part, 'target', target_body_part).alias(f'at__{agent_body_part}__{target_body_part}__distance') for agent_body_part, target_body_part in itertools.product(KEYPOINT_NAMES, repeat=2)])
        feature_expressions.extend([add_part_speed('agent', keypoint_name, time_lag_ms).alias(f'agent__{keypoint_name}__speed_{time_lag_ms}ms') for keypoint_name, time_lag_ms in itertools.product(speed_keypoints, MODEL_CONFIG['speed_time_scales'])])
        feature_expressions.extend([add_part_speed('target', keypoint_name, time_lag_ms).alias(f'target__{keypoint_name}__speed_{time_lag_ms}ms') for keypoint_name, time_lag_ms in itertools.product(speed_keypoints, MODEL_CONFIG['speed_time_scales'])])
        feature_expressions.extend([add_body_elongation('agent').alias('agent__elongation'), add_body_elongation('target').alias('target__elongation'), add_body_angle('agent').alias('agent__body_angle'), add_body_angle('target').alias('target__body_angle')])
        for agg in ['mean', 'std', 'min', 'max']:
            for time_lag_ms in scales:
                feature_expressions.append(add_center_distance_rolling(agg, time_lag_ms).alias(f'at__body_center__dist_{agg}_{time_lag_ms}ms'))
        for time_lag_ms in dist_change_time_scales:
            feature_expressions.append(add_distance_change_rate('body_center', 'body_center', time_lag_ms).alias(f'at__body_center__dist_change_rate_{time_lag_ms}ms'))
        key_target_parts = ['nose', 'body_center', 'tail_base']
        for target_part in key_target_parts:
            for time_lag_ms in scales:
                feature_expressions.append(add_distance_change_rate('nose', target_part, time_lag_ms).alias(f'at__nose__{target_part}__dist_change_rate_{time_lag_ms}ms'))
        for agg in ['mean', 'std']:
            for time_lag_ms in scales:
                feature_expressions.append(add_distance_change_rolling('body_center', 'body_center', time_lag_ms, agg).alias(f'at__body_center__dist_change_{agg}_{time_lag_ms}ms'))
        for time_lag_ms in scales:
            feature_expressions.append(add_approaching_ratio('body_center', 'body_center', time_lag_ms).alias(f'at__body_center__approaching_ratio_{time_lag_ms}ms'))
        for time_lag_ms in scales:
            feature_expressions.append(add_distance_change_rolling('body_center', 'body_center', time_lag_ms, 'min').alias(f'at__body_center__dist_change_min_{time_lag_ms}ms'))
            feature_expressions.append(add_distance_change_rolling('body_center', 'body_center', time_lag_ms, 'max').alias(f'at__body_center__dist_change_max_{time_lag_ms}ms'))
        feature_expressions.extend([add_agent_facing_target().alias('at__agent_facing_target'), add_target_facing_agent().alias('at__target_facing_agent'), (add_agent_facing_target() * add_target_facing_agent()).alias('at__mutual_facing'), add_agent_behind_target().alias('at__agent_behind_target'), add_agent_nose_to_rear().alias('at__agent_nose_to_target_rear')])
        for time_lag_ms in scales:
            feature_expressions.extend([add_relative_speed(time_lag_ms).alias(f'at__relative_speed_{time_lag_ms}ms'), add_speed_ratio(time_lag_ms).alias(f'at__speed_ratio_{time_lag_ms}ms')])
        feature_columns = merged_pivot.with_columns(pl.lit(agent_id).alias('agent_mouse_id'), pl.lit(target_id).alias('target_mouse_id')).select(feature_expressions)
        segment_record = segment_record.join(feature_columns, on=['video_frame', 'agent_mouse_id', 'target_mouse_id'], how='left')
        result_df.append(segment_record)
    return pl.concat(result_df, how='vertical')

# 解析测试集中每个视频允许预测的行为列表。
def parse_labeled_actions(behaviors_text):
    if behaviors_text is None:
        return []
    return ast.literal_eval(behaviors_text)

# 根据测试集元数据展开需要推理的 video-agent-target 任务。
def build_prediction_task_table(test_meta_df: pl.DataFrame) -> pl.DataFrame:
    return test_meta_df.filter(pl.col('behaviors_labeled').is_not_null()).select(['lab_id', 'video_id', 'behaviors_labeled']).with_columns(pl.col('behaviors_labeled').map_elements(parse_labeled_actions, return_dtype=pl.List(pl.Utf8)).alias('behaviors_labeled_list')).explode('behaviors_labeled_list').rename({'behaviors_labeled_list': 'behaviors_labeled_element'}).with_columns(pl.col('behaviors_labeled_element').str.split(',').list.get(0).str.replace_all("[()' ]", '').alias('agent'), pl.col('behaviors_labeled_element').str.split(',').list.get(1).str.replace_all("[()' ]", '').alias('target'), pl.col('behaviors_labeled_element').str.split(',').list.get(2).str.replace_all("[()' ]", '').alias('behavior')).select(['lab_id', 'video_id', 'agent', 'target', 'behavior'])

# 从字符串形式的鼠标编号中提取数字 ID。
def parse_mouse_number(mouse_text: str) -> int:
    if mouse_text == 'self':
        return -1
    m = re.search('mouse(\\d+)', mouse_text)
    if m:
        return int(m.group(1))
    raise ValueError(f'Unexpected mouse id format: {mouse_text}')

# 对逐帧预测概率进行时间平滑，降低帧级抖动。
def smooth_time_probabilities(probabilities: np.ndarray, decision_threshold: float, window_size: int=5, min_segment_length: int=5) -> np.ndarray:
    """概率平滑和阈值应用"""
    if window_size > 1:
        smooth_kernel = np.ones(window_size) / window_size
        probabilities = np.convolve(probabilities, smooth_kernel, mode='same')
    binary = (probabilities >= decision_threshold).astype(np.int8)
    if min_segment_length > 1:
        diff = np.diff(np.concatenate([[0], binary, [0]]))
        starts = np.where(diff == 1)[0]
        ends = np.where(diff == -1)[0]
        for s, e in zip(starts, ends):
            if e - s < min_segment_length:
                binary[s:e] = 0
    return binary

# 加载指定实验室与行为对应的模型集合和阈值。
def load_action_model_ensemble(lab_name: str, action_name: str):
    """加载XGBoost和LightGBM模型"""
    action_model_dir = TRAINED_MODEL_DIR / 'results' / lab_name / action_name
    if not action_model_dir.exists():
        return ([], [], 0.5)
    xgb_model_list = []
    lgb_model_list = []
    fold_thresholds = []
    for fold_id in range(3):
        fold_dir = action_model_dir / f'fold_{fold_id}'
        xgb_model_path = fold_dir / 'xgb_model.json'
        if xgb_model_path.exists():
            estimator = xgb.Booster()
            estimator.load_model(str(xgb_model_path))
            xgb_model_list.append(estimator)
        lgb_model_path = fold_dir / 'lgb_model.txt'
        if lgb_model_path.exists():
            estimator = lgb.Booster(model_file=str(lgb_model_path))
            lgb_model_list.append(estimator)
        thr_path = fold_dir / 'threshold.txt'
        if thr_path.exists():
            with open(thr_path, 'r') as f:
                fold_thresholds.append(float(f.read().strip()))
    xgb_model_path = action_model_dir / 'full_train' / 'xgb_model.json'
    if xgb_model_path.exists():
        estimator = xgb.Booster()
        estimator.load_model(str(xgb_model_path))
        xgb_model_list.append(estimator)
    else:
        print('全数据模型没有加载成功')
    mean_threshold = np.mean(fold_thresholds) if fold_thresholds else 0.5
    return (xgb_model_list, lgb_model_list, mean_threshold)

# 对单个 video-agent-target 任务生成行为片段预测。
def predict_action_segments(lab_name: str, video_key: int, agent_role: str, target_role: str, behaviors: list, index_df: pl.DataFrame, feature_df: pl.DataFrame):
    """对一个group进行预测"""
    if feature_df.height == 0:
        return None
    num_samples = feature_df.height
    model_feature_names = list(feature_df.columns)
    feature_matrix = feature_df.to_pandas()
    action_probability_map = {}
    action_threshold_map = {}
    for action_name in behaviors:
        xgb_model_list, lgb_model_list, decision_threshold = load_action_model_ensemble(lab_name, action_name)
        if not xgb_model_list:
            continue
        xgb_probabilities = np.zeros(num_samples, dtype=np.float32)
        dtest = xgb.DMatrix(feature_matrix, feature_names=model_feature_names)
        for estimator in xgb_model_list:
            xgb_probabilities += estimator.predict(dtest)
        xgb_probabilities /= len(xgb_model_list)
        if lgb_model_list and MODEL_CONFIG['use_ensemble']:
            lgb_probabilities = np.zeros(num_samples, dtype=np.float32)
            for estimator in lgb_model_list:
                lgb_probabilities += estimator.predict(feature_matrix)
            lgb_probabilities /= len(lgb_model_list)
            blended_probabilities = (xgb_probabilities + lgb_probabilities) / 2
        else:
            blended_probabilities = xgb_probabilities
        if MODEL_CONFIG['smoothing_window'] > 1:
            smooth_kernel = np.ones(MODEL_CONFIG['smoothing_window']) / MODEL_CONFIG['smoothing_window']
            blended_probabilities = np.convolve(blended_probabilities, smooth_kernel, mode='same')
        action_probability_map[action_name] = blended_probabilities
        action_threshold_map[action_name] = decision_threshold
    if not action_probability_map:
        return None
    selected_actions = []
    for i in range(num_samples):
        selected_action = 'none'
        selected_score = 0
        for beh, probabilities in action_probability_map.items():
            thr = action_threshold_map[beh]
            if probabilities[i] >= thr and probabilities[i] > selected_score:
                selected_action = beh
                selected_score = probabilities[i]
        selected_actions.append(selected_action)
    prediction_segments_df = index_df.with_columns(pl.Series('prediction', selected_actions))
    agent_id = parse_mouse_number(agent_role)
    target_id = parse_mouse_number(target_role)
    task_submission_df = prediction_segments_df.filter(pl.col('prediction') != pl.col('prediction').shift(1)).with_columns(pl.col('video_frame').shift(-1).alias('stop_frame')).filter(pl.col('prediction') != 'none').select(pl.col('video_id'), (pl.lit('mouse') + pl.lit(agent_id).cast(pl.Utf8)).alias('agent_id'), pl.when(pl.lit(target_id) == -1).then(pl.lit('self')).otherwise(pl.lit('mouse') + pl.lit(target_id).cast(pl.Utf8)).alias('target_id'), pl.col('prediction').alias('action'), pl.col('video_frame').alias('start_frame'), pl.col('stop_frame'))
    return task_submission_df

# 对预测片段进行后处理，修正异常区间并过滤无效结果。
def clean_submission_segments(submission_df: pl.DataFrame, test_meta_df: pl.DataFrame) -> pl.DataFrame:
    """后处理：过滤无效片段"""
    submission_df = submission_df.filter(pl.col('start_frame') < pl.col('stop_frame'))
    submission_df = submission_df.filter(pl.col('stop_frame') - pl.col('start_frame') >= MODEL_CONFIG['min_duration'])
    return submission_df
if __name__ == '__main__':
    print('=' * 60)
    print('MABe Improved Training - Inference')
    print('=' * 60)
    print('\n[1/4] 加载测试数据...')
    test_meta_df = pl.read_csv(DATA_ROOT / 'test.csv')
    prediction_task_df = build_prediction_task_table(test_meta_df)
    behavior_groups = list(prediction_task_df.group_by('lab_id', 'video_id', 'agent', 'target', maintain_order=True))
    print(f'共 {len(behavior_groups)} 个 (lab, video, agent, target) 组合')
    print('\n[2/4] 生成特征...')
    video_rows = test_meta_df.rows(named=True)
    for video_row in tqdm(video_rows, desc='Feature generation'):
        lab_name = video_row['lab_id']
        video_key = video_row['video_id']
        tracking_path = TEST_POSE_DIR / f'{lab_name}/{video_key}.parquet'
        pose_df = pl.read_parquet(tracking_path)
        self_feat = build_self_mouse_features(video_meta=video_row, pose_df=pose_df)
        pair_feat = build_pair_mouse_features(video_meta=video_row, pose_df=pose_df)
        self_feat.write_parquet(SELF_FEATURE_OUTPUT_DIR / f'{video_key}.parquet')
        pair_feat.write_parquet(PAIR_FEATURE_OUTPUT_DIR / f'{video_key}.parquet')
        del self_feat, pair_feat, pose_df
        gc.collect()
    print('\n[3/4] 模型推理...')
    task_submission_tables = []
    for (lab_name, video_key, agent_role, target_role), behavior_group in tqdm(behavior_groups, desc='Inference'):
        agent_id = parse_mouse_number(agent_role)
        target_id = parse_mouse_number(target_role)
        if target_role == 'self':
            feature_file_path = SELF_FEATURE_OUTPUT_DIR / f'{video_key}.parquet'
            working_df = pl.read_parquet(feature_file_path).filter(pl.col('agent_mouse_id') == agent_id)
        else:
            feature_file_path = PAIR_FEATURE_OUTPUT_DIR / f'{video_key}.parquet'
            working_df = pl.read_parquet(feature_file_path).filter((pl.col('agent_mouse_id') == agent_id) & (pl.col('target_mouse_id') == target_id))
        if working_df.height == 0:
            continue
        index_df = working_df.select(KEY_COLUMNS)
        feature_df = working_df.select(pl.exclude(KEY_COLUMNS))
        behaviors = behavior_group.select('behavior').unique()['behavior'].to_list()
        group_sub = predict_action_segments(lab_name=lab_name, video_key=video_key, agent_role=agent_role, target_role=target_role, behaviors=behaviors, index_df=index_df, feature_df=feature_df)
        if group_sub is not None and group_sub.height > 0:
            task_submission_tables.append(group_sub)
    if not task_submission_tables:
        raise RuntimeError('没有生成任何预测！请检查模型是否正确加载。')
    print('\n[4/4] 后处理和保存...')
    submission_df = pl.concat(task_submission_tables, how='vertical')
    print(f'原始预测: {submission_df.height} 行')
    submission_df = clean_submission_segments(submission_df, test_meta_df)
    print(f'后处理后: {submission_df.height} 行')
    submission_df = submission_df.with_row_index('row_id')
    submission_file_path = ARTIFACT_DIR / 'submission.csv'
    submission_df.write_csv(submission_file_path)
    print(f'\n提交文件已保存: {submission_file_path}')
    print('\n提交文件预览:')
    print(submission_df.head(10))
    print('\n完成!')